# NZ Retail Sales — Exploratory Data Analysis

Explores Stats NZ retail, CPI, and employment data fetched via the ADE API.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from forecasting.config import load_config
from forecasting.data import ADEClient

sns.set_theme(style='whitegrid', palette='muted')
cfg = load_config()

## 1. Load Data

In [ ]:
client = ADEClient()
df = client.build_merged_dataset(cfg['data']['start_date'])
print(f'Shape: {df.shape}')
df.head()

## 2. Retail Sales Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['date'], df['retail_sales'], linewidth=1.5)
ax.set_title('NZ Monthly Retail Sales (Stats NZ RSM)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('NZD (millions)')
plt.tight_layout()
plt.show()

## 3. Correlation: Retail vs CPI vs Employment

In [ ]:
corr = df[['retail_sales', 'cpi', 'employment_rate']].corr()
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Pearson Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Seasonal Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

series = df.set_index('date')['retail_sales'].dropna()
result = seasonal_decompose(series, model='additive', period=12)
result.plot()
plt.suptitle('Additive Seasonal Decomposition (period=12)', y=1.02)
plt.tight_layout()
plt.show()

## 5. Year-over-Year Growth

In [ ]:
df['yoy_pct'] = df['retail_sales'].pct_change(12) * 100

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(df['date'], df['yoy_pct'], width=25, color=df['yoy_pct'].apply(lambda x: '#2ca02c' if x >= 0 else '#d62728'))
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Retail Sales YoY Growth (%)')
ax.set_xlabel('Date')
ax.set_ylabel('YoY %')
plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
df[['retail_sales', 'cpi', 'employment_rate']].describe().round(2)